# 02 — Chargement SIRENE

Extrait des parquets INSEE deposes dans `data/raw/sirene/` :

- les **etablissements actifs** avec denomination d'unite legale et enseignes,
  pour la siretisation ;
- les **unites legales actives** avec l'adresse de leur siege, pour la
  sirenisation.

L'ecriture passe par DuckDB, qui streame les donnees sans les charger en
memoire : les parquets INSEE comptent une quinzaine de millions de lignes.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src.connexion import get_duckdb_connection
from src.display import afficher_tableau
from config.settings import SIRENE_ETAB_RAW, SIRENE_UL_RAW, INTERIM_DIR

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
duckdb_con = get_duckdb_connection()
print("Vues SIRENE creees.")

Vues SIRENE creees.


## 1. Etablissements actifs

`activitePrincipaleEtablissement` est ajoutee par rapport a la version
precedente : c'est l'APE au niveau etablissement, compare a celui du referentiel
en phase 3 de siretisation.

In [2]:
duckdb_con.execute(f"""
    COPY (
        SELECT siren, siret, nic,
               denominationUniteLegale, sigleUniteLegale,
               categorieJuridiqueUniteLegale,
               activitePrincipaleUniteLegale, activitePrincipaleEtablissement,
               dateCreationUniteLegale, dateCreationEtablissement,
               etablissementSiege,
               enseigne1Etablissement, enseigne2Etablissement,
               enseigne3Etablissement, denominationUsuelleEtablissement,
               numeroVoieEtablissement, typeVoieEtablissement,
               libelleVoieEtablissement, codeCommuneEtablissement
        FROM etab_active
    ) TO '{SIRENE_ETAB_RAW}' (FORMAT PARQUET)
""")
print(f"Etablissements ecrits -> {SIRENE_ETAB_RAW}")
afficher_tableau(pd.read_parquet(SIRENE_ETAB_RAW).head(), "Apercu Etab SIRENE")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Etablissements ecrits -> /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/data/interim/sirene_etab.parquet


siren,siret,nic,denominationUniteLegale,sigleUniteLegale,categorieJuridiqueUniteLegale,activitePrincipaleUniteLegale,activitePrincipaleEtablissement,dateCreationUniteLegale,dateCreationEtablissement,etablissementSiege,enseigne1Etablissement,enseigne2Etablissement,enseigne3Etablissement,denominationUsuelleEtablissement,numeroVoieEtablissement,typeVoieEtablissement,libelleVoieEtablissement,codeCommuneEtablissement
000325175,00032517500065,00065,None,None,1000,32.12Z,32.12Z,2000-09-26,2018-02-07,True,None,None,None,None,51,RUE,MARX DORMOY,13204
005420021,00542002100056,00056,ETABLISSEMENTS LUCIEN BIQUEZ,None,5710,46.69B,46.69B,1954-01-01,2009-12-23,True,None,None,None,None,21,BOULEVARD,DES PRES,80001
005420120,00542012000015,00015,SOCIETE DES SUCRERIES DU MARQUENTERRE,None,5599,70.10Z,10.81Z,1954-01-01,1989-01-27,False,None,None,None,None,None,RUE,DE LA FONTAINE,80688
005420120,00542012000023,00023,SOCIETE DES SUCRERIES DU MARQUENTERRE,None,5599,70.10Z,52.10B,1954-01-01,1900-01-01,False,None,None,None,None,12,ROUTE,DE MONTREUIL,62688
005420120,00542012000056,00056,SOCIETE DES SUCRERIES DU MARQUENTERRE,None,5599,70.10Z,70.10Z,1954-01-01,2026-01-13,True,None,None,None,None,32,CHEMIN,DES GARENNES,80713


## 2. Unites legales avec adresse du siege

In [3]:
duckdb_con.execute(f"""
    COPY (
        SELECT siren, denominationUniteLegale, sigleUniteLegale,
               categorieJuridiqueUniteLegale, activitePrincipaleUniteLegale,
               dateCreationUniteLegale, nicSiegeUniteLegale,
               numeroVoieEtablissement, typeVoieEtablissement,
               libelleVoieEtablissement, codeCommuneEtablissement
        FROM etab_siege
    ) TO '{SIRENE_UL_RAW}' (FORMAT PARQUET)
""")
duckdb_con.close()
print(f"Unites legales ecrites -> {SIRENE_UL_RAW}")
afficher_tableau(pd.read_parquet(SIRENE_UL_RAW).head(), "Apercu UL SIRENE")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unites legales ecrites -> /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/data/interim/sirene_ul.parquet


siren,denominationUniteLegale,sigleUniteLegale,categorieJuridiqueUniteLegale,activitePrincipaleUniteLegale,dateCreationUniteLegale,nicSiegeUniteLegale,numeroVoieEtablissement,typeVoieEtablissement,libelleVoieEtablissement,codeCommuneEtablissement
000325175,None,None,1000,32.12Z,2000-09-26,00065,51,RUE,MARX DORMOY,13204
005420021,ETABLISSEMENTS LUCIEN BIQUEZ,None,5710,46.69B,1954-01-01,00056,21,BOULEVARD,DES PRES,80001
005420120,SOCIETE DES SUCRERIES DU MARQUENTERRE,None,5599,70.10Z,1954-01-01,00056,32,CHEMIN,DES GARENNES,80713
005440110,None,None,1000,68.20B,1954-01-01,00061,6,RUE,FIRMIN GUICHARD,04070
005450093,BASALPINE AUTOCARS,None,5499,77.11A,1954-01-01,00033,None,AVENUE,DES THERMES,04094
